# Socratia · Qwen propio en GPU de Kaggle/Colab

Este notebook ejecuta pesos abiertos en la GPU de la sesión. No usa Gemini/Groq ni la CPU de la laptop. Es inferencia remota autohospedada, no entrenamiento propio ni un servidor permanente.

Selecciona GPU y habilita Internet para descargar el modelo. No subas documentos privados. El ejemplo usa datos ficticios. Una sesión sin GPU se detiene antes de descargar pesos. No se abre ningún endpoint público.

Fuentes: https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507 y https://www.kaggle.com/docs/notebooks


In [ ]:
import torch
assert torch.cuda.is_available(), "Activa una GPU en la configuración de la sesión. No se ejecutará en CPU."
print(torch.cuda.get_device_name(0))
free,total=torch.cuda.mem_get_info()
assert free>10*1024**3, "Se requieren al menos 10 GiB libres para esta configuración; usa otra GPU."


In [ ]:
%pip install -q "transformers>=4.51,<5" "accelerate>=1,<2"


In [ ]:
import time, json
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL="Qwen/Qwen3-4B-Instruct-2507"
tokenizer=AutoTokenizer.from_pretrained(MODEL)
model=AutoModelForCausalLM.from_pretrained(MODEL,torch_dtype=torch.float16,device_map={"":"cuda:0"})


In [ ]:
contexto="""Proyecto ficticio: agenda de tutorías. La reserva usa identificadores únicos para evitar duplicados. Se evaluaron treinta casos simulados; dos detectaron conflictos de calendario y fueron corregidos. Los veinte participantes son ficticios; no permiten generalizar. No existen resultados reales de una universidad."""
messages=[{"role":"system","content":"Eres evaluador de defensas. Usa solo el contexto dado. Redacta en español y devuelve JSON con pregunta, razon, limitacion. No inventes resultados."},{"role":"user","content":contexto+"\nFormula una pregunta crítica sobre validez de los resultados y explica qué evidencia falta."}]
inputs=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors="pt").to("cuda:0")
started=time.monotonic()
with torch.inference_mode():
    output=model.generate(inputs,max_new_tokens=500,do_sample=False)
text=tokenizer.decode(output[0][inputs.shape[-1]:],skip_special_tokens=True)
report={"model":MODEL,"inference":"self_hosted_gpu","gpu":torch.cuda.get_device_name(0),"seconds":round(time.monotonic()-started,2),"synthetic_context":True,"result":text}
Path("socratia-qwen-evidence.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(report,ensure_ascii=False,indent=2))


## Integración con Socratia

El backend incorpora adaptadores para preguntas y diapositivas compatibles con `/v1/chat/completions`, con validación de JSON, fuentes y español. Para utilizarlos se necesita un servidor GPU accesible de forma segura, con HTTPS y token. Variables privadas del backend: `SELF_HOSTED_LLM_URL`, `SELF_HOSTED_LLM_KEY`, `SELF_HOSTED_LLM_MODEL`; seleccionar `PREPARATION_PRIMARY_PROVIDER=self_hosted`.

Este notebook no promete que Kaggle exponga puertos ni que una sesión dure indefinidamente. No montes túneles sin comprobar las condiciones de la plataforma. Un servidor vLLM privado con autenticación puede servir el mismo modelo; detener la GPU elimina su disponibilidad. Si falla, Socratia conserva la alternativa externa configurada. No declares probado el adaptador contra GPU real hasta ejecutar una generación válida desde AWS.
